In [1]:
import warnings
import gc
from pathlib import Path

import numpy as np
import cv2
import torch
import torch.nn.functional as F
from PIL import Image
from matplotlib import pyplot as plt

from wan import WanI2V
from wan.configs.wan_i2v_14B import i2v_14B
from wan.utils.utils import cache_video

from torchcodec.decoders import VideoDecoder
from torchvision import transforms

warnings.filterwarnings("ignore", category=FutureWarning, message=".*torch.cuda.amp.autocast.*")

In [2]:
wan_i2v = WanI2V(
    config=i2v_14B,
    checkpoint_dir="./weights/Wan2.1-I2V-14B-480P/",
    device_id=0,
    t5_cpu=True,
)

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [3]:
text_encoder = wan_i2v.text_encoder
hf_tokenizer = text_encoder.tokenizer
t5_tokenizer = hf_tokenizer.tokenizer 

In [12]:
len(t5_tokenizer.get_vocab())

256300

In [ ]:
t5_tokenizer.get_added_vocab()

{'<pad>': 0,
 '</s>': 1,
 '<s>': 2,
 '<unk>': 3,
 '<extra_id_299>': 256000,
 '<extra_id_298>': 256001,
 '<extra_id_297>': 256002,
 '<extra_id_296>': 256003,
 '<extra_id_295>': 256004,
 '<extra_id_294>': 256005,
 '<extra_id_293>': 256006,
 '<extra_id_292>': 256007,
 '<extra_id_291>': 256008,
 '<extra_id_290>': 256009,
 '<extra_id_289>': 256010,
 '<extra_id_288>': 256011,
 '<extra_id_287>': 256012,
 '<extra_id_286>': 256013,
 '<extra_id_285>': 256014,
 '<extra_id_284>': 256015,
 '<extra_id_283>': 256016,
 '<extra_id_282>': 256017,
 '<extra_id_281>': 256018,
 '<extra_id_280>': 256019,
 '<extra_id_279>': 256020,
 '<extra_id_278>': 256021,
 '<extra_id_277>': 256022,
 '<extra_id_276>': 256023,
 '<extra_id_275>': 256024,
 '<extra_id_274>': 256025,
 '<extra_id_273>': 256026,
 '<extra_id_272>': 256027,
 '<extra_id_271>': 256028,
 '<extra_id_270>': 256029,
 '<extra_id_269>': 256030,
 '<extra_id_268>': 256031,
 '<extra_id_267>': 256032,
 '<extra_id_266>': 256033,
 '<extra_id_265>': 256034,
 '<ext

In [4]:
wan_i2v.text_encoder.__call__??

Signature: wan_i2v.text_encoder.__call__(texts, device)
Docstring: Call self as a function.
Source:   
    def __call__(self, texts, device):
        ids, mask = self.tokenizer(
            texts, return_mask=True, add_special_tokens=True)
        ids = ids.to(device)
        mask = mask.to(device)
        seq_lens = mask.gt(0).sum(dim=1).long()
        context = self.model(ids, mask)
        return [u[:v] for u, v in zip(context, seq_lens)]
File:      /local_scratch/gzappavi/wan_experiments/wan2.1/wan/modules/t5.py
Type:      method

In [5]:
def find_subsequence(seq, subseq):
    seq_len = len(seq)
    subseq_len = len(subseq)

    if subseq_len > seq_len:
        return -1

    for i in range(seq_len - subseq_len + 1):
        window = seq[i : i + subseq_len]
        if window == subseq:
            return i

    return -1


In [6]:
t5_tokenizer.__call__??

Signature:
t5_tokenizer.__call__(
    text: Union[str, list[str], list[list[str]], NoneType] = None,
    text_pair: Union[str, list[str], list[list[str]], NoneType] = None,
    text_target: Union[str, list[str], list[list[str]], NoneType] = None,
    text_pair_target: Union[str, list[str], list[list[str]], NoneType] = None,
    add_special_tokens: bool = True,
    padding: Union[bool, str, transformers.utils.generic.PaddingStrategy] = False,
    truncation: Union[bool, str, transformers.tokenization_utils_base.TruncationStrategy, NoneType] = None,
    max_length: Optional[int] = None,
    stride: int = 0,
    is_split_into_words: bool = False,
    pad_to_multiple_of: Optional[int] = None,
    padding_side: Optional[str] = None,
    return_tensors: Union[str, transformers.utils.generic.TensorType, NoneType] = None,
    return_token_type_ids: Optional[bool] = None,
    return_attention_mask: Optional[bool] = None,
    return_overflowing_tokens: bool = False,
    return_special_tokens_mas

In [7]:
hf_tokenizer(a, padding=False, add_special_tokens=False, return_tensors=None)[0]


NameError: name 'a' is not defined

In [ ]:
a = "New York"
b = "a city"
text = f"{a} is {b}"


text_token_ids = hf_tokenizer(text)[0].tolist()
a_token_ids = t5_tokenizer.encode(a, add_special_tokens=False)
b_token_ids = t5_tokenizer.encode(b, add_special_tokens=False)

start_index_a = find_subsequence(text_token_ids, a_token_ids)
start_index_b = find_subsequence(text_token_ids, b_token_ids)

In [ ]:
hf_tokenizer.__call__??

Signature: hf_tokenizer.__call__(sequence, **kwargs)
Docstring: Call self as a function.
Source:   
    def __call__(self, sequence, **kwargs):
        return_mask = kwargs.pop('return_mask', False)

        # arguments
        _kwargs = {'return_tensors': 'pt'}
        if self.seq_len is not None:
            _kwargs.update({
                'padding': 'max_length',
                'truncation': True,
                'max_length': self.seq_len
            })
        _kwargs.update(**kwargs)

        # tokenization
        if isinstance(sequence, str):
            sequence = [sequence]
        if self.clean:
            sequence = [self._clean(u) for u in sequence]
        ids = self.tokenizer(sequence, **_kwargs)

        # output
        if return_mask:
            return ids.input_ids, ids.attention_mask
        else:
            return ids.input_ids
File:      /local_scratch/gzappavi/wan_experiments/wan2.1/wan/modules/tokenizers.py
Type:      method

In [ ]:
text = "New York is a city"

encoding = t5_tokenizer(
      text,
      return_tensors="pt",
      # padding="max_length",
      # truncation=True,
      # max_length=hf_tokenizer.seq_len,
      # return_offsets_mapping=True,
      add_special_tokens=False,
)
encoding


{'input_ids': tensor([[1373, 3248,  339,  289, 8517]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}

In [ ]:
encoding

{'input_ids': tensor([[1373, 3248,  339,  289, 8517]]), 'attention_mask': tensor([[1, 1, 1, 1, 1]])}

In [ ]:
t5_tokenizer.encode(
    text,
    padding="max_length",
    truncation=True,
    max_length=hf_tokenizer.seq_len,
    add_special_tokens=False)

[1373,
 3248,
 339,
 289,
 8517,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,

In [ ]:
t5_tokenizer.encode??

Signature:
t5_tokenizer.encode(
    text: Union[str, list[str], list[int]],
    text_pair: Union[str, list[str], list[int], NoneType] = None,
    add_special_tokens: bool = True,
    padding: Union[bool, str, transformers.utils.generic.PaddingStrategy] = False,
    truncation: Union[bool, str, transformers.tokenization_utils_base.TruncationStrategy, NoneType] = None,
    max_length: Optional[int] = None,
    stride: int = 0,
    padding_side: Optional[str] = None,
    return_tensors: Union[str, transformers.utils.generic.TensorType, NoneType] = None,
    **kwargs,
) -> list[int]
Docstring:
Converts a string to a sequence of ids (integer), using the tokenizer and vocabulary.

Same as doing `self.convert_tokens_to_ids(self.tokenize(text))`.

Args:
    text (`str`, `list[str]` or `list[int]`):
        The first sequence to be encoded. This can be a string, a list of strings (tokenized string using the
        `tokenize` method) or a list of integers (tokenized string ids using the `conver

In [ ]:
hf_tokenizer.encode

AttributeError: 'HuggingfaceTokenizer' object has no attribute 'encode'

In [ ]:
list(encoding.keys())

['input_ids', 'attention_mask', 'offset_mapping']

In [ ]:
text = "New York is a city"

encoding = tokenizer(text, return_tensors="pt", return_mask=True)

In [ ]:
encoding[1]

tensor([[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0